# BackgroundFX (Gradio Version) - Google Colab Launcher

This notebook runs the new Gradio version of your BackgroundFX app and saves videos to Google Drive.

## Instructions:

1.  **Run `Step 1`** to install all required libraries, including Gradio and the large SAM2 model dependencies. This may take a few minutes.
2.  **Run `Step 2`** to connect your Google Drive. You must authorize access when prompted.
3.  **Run `Step 3`** which contains the full Python code for your app.
4.  **Get a free `ngrok` authtoken** from [your ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
5.  **Run `Step 4`**. Paste your `ngrok` authtoken when prompted.
6.  **Click the public URL** that appears to open your app. Processed videos will be saved to a `BackgroundFX_Output` folder in your Google Drive.

In [ ]:
# Step 1: Install Dependencies for Gradio & AI Models
!pip install gradio opencv-python-headless Pillow requests pyngrok -q
!pip install git+https://github.com/facebookresearch/segment-anything-2.git -q

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 3: Write the Gradio App Code to a File
%%writefile gradio_app.py

import gradio as gr
import os
import cv2
import numpy as np
import requests
from PIL import Image
from io import BytesIO
import tempfile
import shutil
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check if running in Colab to set the correct path
GDRIVE_OUTPUT_DIR = '/content/drive/MyDrive/BackgroundFX_Output'
os.makedirs(GDRIVE_OUTPUT_DIR, exist_ok=True)

# --- AI Model Availability & Loading ---
SAM2_AVAILABLE = False
SAM2_PREDICTOR = None

try:
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    SAM2_PREDICTOR = SAM2ImagePredictor.from_pretrained("facebook/sam2-hiera-large")
    SAM2_AVAILABLE = True
    logger.info("✅ SAM2 (Large Model) loaded successfully")
except ImportError:
    logger.warning("⚠️ SAM2 not available. Please ensure it's installed.")
except Exception as e:
    logger.error(f'🚨 Error loading SAM2 model: {e}')

# --- Core Processing Logic ---
def get_background_options():
    return {
        "Brick Wall": "https://images.unsplash.com/photo-1558618666-fcd25c85cd64?w=1280&h=720&fit=crop",
        "Simple Office": "https://images.unsplash.com/photo-1497366216548-37526070297c?w=1280&h=720&fit=crop",
        "Executive Office": "https://images.unsplash.com/photo-1586953208448-b95a79798f07?w=1280&h=720&fit=crop",
        "Modern Conference Room": "https://images.unsplash.com/photo-1560472354-b33ff0c44a43?w=1280&h=720&fit=crop",
    }

def segment_person_sam2(frame):
    if not SAM2_AVAILABLE or SAM2_PREDICTOR is None:
        return None
    try:
        SAM2_PREDICTOR.set_image(frame)
        h, w = frame.shape[:2]
        center_point = np.array([[w // 2, h // 2]])
        center_label = np.array([1])
        masks, _, _ = SAM2_PREDICTOR.predict(point_coords=center_point, point_labels=center_label, multimask_output=False)
        return masks[0] if len(masks) > 0 else None
    except Exception as e:
        logger.error(f'SAM2 segmentation failed: {e}')
        return None

def chroma_key_replacement(original_frame, person_mask, new_background):
    h, w, _ = original_frame.shape
    background_resized = cv2.resize(new_background, (w, h))
    # Ensure mask is boolean and can be used for indexing
    mask_bool = person_mask.astype(bool)
    # Create a 3-channel mask for element-wise multiplication
    mask_3d = np.stack([mask_bool]*3, axis=-1)
    # Combine the person from the original frame and the new background
    output_frame = np.where(mask_3d, original_frame, background_resized)
    return output_frame.astype(np.uint8)

# --- Main Processing Function for Gradio ---
def process_video(video_path, background_choice, custom_bg_image, progress=gr.Progress(track_tqdm=True)):
    if video_path is None:
        raise gr.Error("Please upload a video first.")

    if background_choice == "Custom" and custom_bg_image is None:
        raise gr.Error("Please upload a custom background image or choose a preset.")

    # Load background
    if background_choice == "Custom":
        background_image = np.array(custom_bg_image)
    else:
        background_url = get_background_options()[background_choice]
        background_image = np.array(Image.open(BytesIO(requests.get(background_url).content)))

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    temp_output_path = tempfile.mktemp(suffix='.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_output_path, fourcc, fps, (width, height))

    for i in progress.tqdm(range(total_frames), desc="Processing Frames"):
        ret, frame = cap.read()
        if not ret: break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        person_mask = segment_person_sam2(frame_rgb)
        if person_mask is not None:
            final_frame = chroma_key_replacement(frame_rgb, person_mask, background_image)
        else:
            final_frame = frame_rgb # Fallback if segmentation fails
        out.write(cv2.cvtColor(final_frame, cv2.COLOR_RGB2BGR))

    cap.release()
    out.release()

    # Save to Google Drive and return path
    final_filename = f"processed_{os.path.basename(video_path)}"
    drive_path = os.path.join(GDRIVE_OUTPUT_DIR, final_filename)
    shutil.copyfile(temp_output_path, drive_path)
    
    status_message = f"✅ Processing complete! Video saved to: {drive_path}"
    logger.info(status_message)
    
    return temp_output_path, status_message

# --- Gradio UI Definition ---
with gr.Blocks(theme=gr.themes.Soft(), title="BackgroundFX") as demo:
    gr.Markdown("## 🎬 BackgroundFX - AI Video Background Replacement")
    gr.Markdown("Upload a video, choose a new background, and let the AI do the rest. Results are saved to your Google Drive.")

    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label="Upload Your Video")
            background_options = list(get_background_options().keys())
            background_choice = gr.Dropdown(background_options + ["Custom"], label="Choose a Background", value=background_options[0])
            custom_bg_image = gr.Image(type="pil", label="Upload Custom Background", visible=False)
            process_button = gr.Button("🎬 Process Video", variant="primary")

        with gr.Column(scale=1):
            video_output = gr.Video(label="Processed Video")
            status_output = gr.Textbox(label="Status", interactive=False)

    def toggle_custom_bg(choice):
        return gr.update(visible=choice == "Custom")

    background_choice.change(fn=toggle_custom_bg, inputs=background_choice, outputs=custom_bg_image)
    
    process_button.click(
        fn=process_video,
        inputs=[video_input, background_choice, custom_bg_image],
        outputs=[video_output, status_output]
    )

# This check is not strictly necessary in Colab but good practice
if __name__ == "__main__":
    demo.launch(debug=True)

In [ ]:
# Step 4: Run the App with ngrok
from pyngrok import ngrok
import getpass

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
authtoken = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(authtoken)

# Run the Gradio app in the background and expose it with ngrok
public_url = ngrok.connect(7860) # Gradio's default port is 7860
print(f'

🚀 Click here to open your app: {public_url}')
!python gradio_app.py